In [ ]:
import os
import requests
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr
import json
from content_extractor import extract_text

In [ ]:
load_dotenv(override=True)
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
if GROQ_API_KEY is None or GROQ_API_KEY.strip() == "":
    raise ValueError("GROQ_API_KEY is not set in the environment variables.")
else:
    print("Groq API Key found")

In [ ]:
openai = OpenAI(api_key=GROQ_API_KEY,base_url="https://api.groq.com/openai/v1")

In [ ]:
SYSTEM_PROMPT = "You are a snarky assistant that summarizes text."

In [ ]:
url = "https://migrationology.com/pakistani-street-food-karachi/"
responseText = extract_text(url)

In [ ]:
USER_PROMPT = f"Summarize the following text:' {responseText}'"

In [ ]:
foods = [
    {
        "name": "Nihari",
        "category": "Breakfast",
        "price": 700
    },
    {
        "name": "Arabian Paratha",
        "category": "Breakfast",
        "price": 450
    },
    {
        "name": "Kulfi (Clay Cup)",
        "category": "Dessert",
        "price": 250
    },
    {
        "name": "Rabri",
        "category": "Dessert",
        "price": 350
    },
    {
        "name": "Bone Marrow Biryani",
        "category": "Main Course",
        "price": 1200
    },
    {
        "name": "Ninja Street Salad",
        "category": "Street Food",
        "price": 300
    },
    {
        "name": "Fish Kata-Kat",
        "category": "Main Course",
        "price": 900
    },
    {
        "name": "Bun Kebab",
        "category": "Street Food",
        "price": 180
    },
    {
        "name": "Chicken Curry",
        "category": "Main Course",
        "price": 650
    }
]

In [ ]:
def getPriceOfFood(food_name):
    for food in foods:
        if food["name"].lower() == food_name.lower():
            return f"the price of {food_name} is {food['price']} PKR."

In [ ]:
price_function = {
    "name":"getPriceOfFood",
    "description":"Get the price of a food item from collection of foods",
    "paramters":{
        "type":"object",
        "properties":{
            "food_name":{
                "type":"string",
                "description":"the name of the food item to get the price for"
            }
        }
    },
    "required":["food_name"],
    "additionalProperties":False
}

In [ ]:
tools = [{"role":"function","function":getPriceOfFood}]

In [ ]:
def handle_tool_call(message):
    response = []
    for tools in message.tool_calls:
        if tools.function.name == "getPriceOfFood":
            arguments = json.loads(tools.function.arguments)
            food_name = arguments.get("food_name")
            result = getPriceOfFood(food_name)
            response.append({
                "role":"tool",
                "content":result,
                "tool_call_id":tools.id
            })
    return response;

In [ ]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":SYSTEM_PROMPT}] + history + [{"role":"user","content":message}]
    response = openai.chat.completions.create(model="llama-3.1-8b-instant", messages=messages,tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        toolCallData = handle_tool_call(message)
        messages.append(message)
        messages.extend(toolCallData)
        response = openai.chat.completions.create(model="llama-3.1-8b-instant",messages=messages)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat,type="messages").launch()